
**Tasks implemented:**
- **Task 1** — Smart Product Recommendation Engine (complementary products)
- **Task 2** — Unique Product Catalog Creation (deduplication via clustering)
- **Task 3** — Reverse Product Search (text → image retrieval)

**Backbone:** OpenAI CLIP (`ViT-B/32`) — shared embedding space for images and text.

---

## Setup & Installation

In [ ]:
# Install dependencies
!pip install -q git+https://github.com/openai/CLIP.git
!pip install -q kaggle ipywidgets

In [ ]:
import os, json, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.gridspec import GridSpec
import torch
from PIL import Image
from pathlib import Path
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.cluster import AgglomerativeClustering
from scipy.cluster.hierarchy import dendrogram, linkage
from tqdm.notebook import tqdm
import ipywidgets as widgets
from IPython.display import display, clear_output
import clip

warnings.filterwarnings('ignore')

# Device
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {device}')

## Dataset Download

> **Setup:** Upload your `kaggle.json` API token when prompted.  
> Get it from: https://www.kaggle.com/settings → API → Create New Token

In [ ]:
from google.colab import files

print('Upload your kaggle.json API token:')
uploaded = files.upload()

!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json
print('Kaggle API configured ✓')

In [ ]:
# Download the small version (~280MB)
!kaggle datasets download -d paramaggarwal/fashion-product-images-small
!unzip -q fashion-product-images-small.zip -d fashion_data
print('Dataset downloaded and extracted ✓')
!ls fashion_data/

## Core Setup: Load CLIP & Dataset

In [ ]:
# ── Load CLIP ──────────────────────────────────────────────────────────────────
model, preprocess = clip.load('ViT-B/32', device=device)
model.eval()
print(f'CLIP ViT-B/32 loaded on {device} ✓')

In [ ]:
# ── Load styles.csv ────────────────────────────────────────────────────────────
DATA_DIR   = Path('fashion_data')
IMAGES_DIR = DATA_DIR / 'images'

styles = pd.read_csv(DATA_DIR / 'styles.csv', on_bad_lines='skip')

# Keep only rows whose image actually exists
styles['img_path'] = styles['id'].apply(lambda x: IMAGES_DIR / f'{x}.jpg')
styles = styles[styles['img_path'].apply(lambda p: p.exists())].reset_index(drop=True)

print(f'Total products with images: {len(styles)}')
print(f'Columns: {list(styles.columns)}')
styles[['id','masterCategory','subCategory','articleType','productDisplayName']].head(8)

In [ ]:
# ── Subsample for speed (change N_SAMPLES to use more) ────────────────────────
# Using 1000 products gives a good demo; set to len(styles) for full dataset
N_SAMPLES = 1000

# Stratified sample — keep category balance
sample = (
    styles.groupby('masterCategory', group_keys=False)
    .apply(lambda g: g.sample(min(len(g), max(1, int(N_SAMPLES * len(g) / len(styles))))))
    .reset_index(drop=True)
)
sample = sample.sample(min(N_SAMPLES, len(sample))).reset_index(drop=True)

print(f'Working with {len(sample)} products')
print(sample['masterCategory'].value_counts().to_string())

## CLIP Embedding Pipeline

Generate 512-d image embeddings for every product. This is computed once and reused across all three tasks.

In [ ]:
def get_image_embeddings(df: pd.DataFrame, batch_size: int = 64) -> np.ndarray:
    """Embed all product images with CLIP. Returns normalised (N, 512) array."""
    all_embeds = []

    for start in tqdm(range(0, len(df), batch_size), desc='Embedding images'):
        batch_rows = df.iloc[start : start + batch_size]
        images = []
        for _, row in batch_rows.iterrows():
            try:
                img = preprocess(Image.open(row['img_path']).convert('RGB'))
            except Exception:
                img = preprocess(Image.new('RGB', (224, 224), color=(200, 200, 200)))
            images.append(img)

        batch_tensor = torch.stack(images).to(device)
        with torch.no_grad():
            features = model.encode_image(batch_tensor).float()

        # L2-normalise so dot product == cosine similarity
        features = features / features.norm(dim=-1, keepdim=True)
        all_embeds.append(features.cpu().numpy())

    return np.vstack(all_embeds)   # (N, 512)


# Compute embeddings (takes ~1-2 min on GPU for 1000 images)
embeddings = get_image_embeddings(sample)
print(f'Embedding matrix shape: {embeddings.shape}')
print(f'Each vector L2-norm (should be 1.0): {np.linalg.norm(embeddings[0]):.4f}')

---
# Task 1 — Smart Product Recommendation Engine

**Goal:** Given a product, recommend complementary items (things you'd buy *with* it, not just similar-looking items).

**Approach:**
1. Map each `masterCategory` to the categories that naturally complement it (domain knowledge)
2. Compute cosine similarity between the query embedding and all candidate embeddings in those categories
3. Return the top-k highest-scoring candidates

In [ ]:
# ── Complementary category map ─────────────────────────────────────────────────
# Based on fashion domain knowledge — what gets worn / bought together
COMPLEMENTARY_MAP = {
    'Apparel':          ['Footwear', 'Accessories'],
    'Footwear':         ['Apparel', 'Accessories'],
    'Accessories':      ['Apparel', 'Footwear'],
    'Personal Care':    ['Apparel', 'Accessories'],
    'Free Items':       ['Apparel', 'Accessories'],
    'Sporting Goods':   ['Footwear', 'Accessories'],
    'Home':             ['Personal Care'],
}

# Sub-category refinement (article-type level)
ARTICLE_COMPLEMENT_MAP = {
    'Tshirts':          ['Jeans', 'Casual Shoes', 'Watches', 'Sunglasses'],
    'Shirts':           ['Trousers', 'Formal Shoes', 'Belts', 'Watches'],
    'Jeans':            ['Tshirts', 'Casual Shoes', 'Belts'],
    'Casual Shoes':     ['Jeans', 'Tshirts', 'Socks'],
    'Formal Shoes':     ['Shirts', 'Trousers', 'Belts'],
    'Sports Shoes':     ['Track Pants', 'Socks', 'Sports Accessories'],
    'Watches':          ['Shirts', 'Tshirts', 'Formal Shoes'],
    'Sunglasses':       ['Tshirts', 'Casual Shoes'],
    'Kurtas':           ['Churidar', 'Flats', 'Dupatta'],
    'Sarees':           ['Heels', 'Clutches', 'Earrings'],
    'Dresses':          ['Heels', 'Sandals', 'Clutches', 'Earrings'],
    'Tops':             ['Jeans', 'Skirts', 'Flats', 'Sandals'],
    'Heels':            ['Dresses', 'Tops', 'Skirts'],
    'Flats':            ['Kurtas', 'Tops', 'Jeans'],
    'Bags':             ['Tops', 'Dresses', 'Casual Shoes'],
    'Backpacks':        ['Tshirts', 'Casual Shoes', 'Jeans'],
    'Belts':            ['Jeans', 'Trousers', 'Shirts'],
    'Socks':            ['Sports Shoes', 'Casual Shoes'],
    'Track Pants':      ['Sports Shoes', 'Socks'],
    'Jackets':          ['Tshirts', 'Jeans', 'Casual Shoes'],
    'Sweatshirts':      ['Track Pants', 'Sports Shoes'],
}

print('Complementary maps defined ✓')
print(f'Article-level mappings: {len(ARTICLE_COMPLEMENT_MAP)} article types covered')

In [ ]:
def get_complementary_recommendations(query_idx: int, top_k: int = 3):
    """
    For the product at `query_idx` in `sample`, find top_k complementary products.

    Strategy:
    - Try fine-grained articleType map first
    - Fall back to masterCategory map if articleType not in map
    - Score candidates by cosine similarity to query embedding
    """
    query_row    = sample.iloc[query_idx]
    query_embed  = embeddings[query_idx]          # (512,)
    article_type = query_row.get('articleType', '')
    master_cat   = query_row.get('masterCategory', '')

    # Determine target article types
    target_articles = ARTICLE_COMPLEMENT_MAP.get(article_type)
    if target_articles:
        candidates_mask = sample['articleType'].isin(target_articles)
    else:
        target_cats    = COMPLEMENTARY_MAP.get(master_cat, [])
        candidates_mask = sample['masterCategory'].isin(target_cats)

    # Exclude the query product itself
    candidates_mask.iloc[query_idx] = False
    cand_indices = np.where(candidates_mask)[0]

    if len(cand_indices) == 0:
        # Fallback: return most similar across entire catalog (excluding same article type)
        cand_indices = np.array([i for i in range(len(sample))
                                 if i != query_idx and sample.iloc[i]['articleType'] != article_type])

    cand_embeds = embeddings[cand_indices]         # (M, 512)
    scores      = cand_embeds @ query_embed        # cosine sim (already L2-normed)
    top_local   = scores.argsort()[-top_k:][::-1]
    top_global  = cand_indices[top_local]

    results = []
    for idx, score in zip(top_global, scores[top_local]):
        r = sample.iloc[idx].to_dict()
        r['score'] = float(score)
        r['idx']   = idx
        results.append(r)
    return query_row.to_dict(), results


print('Recommendation function defined ✓')

In [ ]:
def visualize_recommendations(query_idx: int, top_k: int = 4):
    query, recs = get_complementary_recommendations(query_idx, top_k=top_k)

    fig, axes = plt.subplots(1, top_k + 1, figsize=(4 * (top_k + 1), 5))
    fig.patch.set_facecolor('#f8f4ff')

    # ── Query image ────────────────────────────────────────────────────────────
    ax = axes[0]
    try:
        ax.imshow(Image.open(query['img_path']).convert('RGB'))
    except Exception:
        ax.text(0.5, 0.5, 'No image', ha='center', va='center', transform=ax.transAxes)
    ax.set_title(f"🔍 Query\n{query.get('productDisplayName','')[:28]}\n({query.get('articleType','')})",
                 fontsize=8, fontweight='bold', color='#4A148C')
    ax.axis('off')
    for spine in ax.spines.values():
        spine.set_edgecolor('#4A148C')
        spine.set_linewidth(3)
    ax.set_frame_on(True)

    # ── Recommended images ─────────────────────────────────────────────────────
    colors = ['#E8F5E9', '#E3F2FD', '#FFF8E1', '#FCE4EC']
    for i, rec in enumerate(recs):
        ax = axes[i + 1]
        try:
            ax.imshow(Image.open(rec['img_path']).convert('RGB'))
        except Exception:
            ax.text(0.5, 0.5, 'No image', ha='center', va='center', transform=ax.transAxes)
        ax.set_title(
            f"#{i+1} {rec.get('productDisplayName','')[:25]}\n"
            f"({rec.get('articleType','')})\nScore: {rec['score']:.3f}",
            fontsize=7.5, color='#1B5E20'
        )
        ax.axis('off')
        fig.axes[i + 1].patch.set_facecolor(colors[i % len(colors)])

    plt.suptitle(
        f'Complementary Recommendations for: {query.get("productDisplayName","")[:40]}',
        fontsize=11, fontweight='bold', color='#4A148C', y=1.02
    )
    plt.tight_layout()
    plt.savefig('task1_recommendations.png', dpi=150, bbox_inches='tight',
                facecolor=fig.get_facecolor())
    plt.show()
    print(f"\n✅ Saved → task1_recommendations.png")


# Demo with a few different product types
for category in ['Footwear', 'Apparel', 'Accessories']:
    cat_indices = sample.index[sample['masterCategory'] == category].tolist()
    if cat_indices:
        print(f'\n--- {category} example ---')
        visualize_recommendations(cat_indices[0], top_k=4)

In [ ]:
# ── Interactive recommendation widget ──────────────────────────────────────────
idx_slider = widgets.IntSlider(
    value=0, min=0, max=len(sample)-1, step=1,
    description='Product #:', style={'description_width': 'initial'},
    layout=widgets.Layout(width='500px')
)
k_slider = widgets.IntSlider(
    value=4, min=1, max=6, step=1,
    description='Top-K:', style={'description_width': 'initial'}
)
btn = widgets.Button(description='🔍 Get Recommendations',
                     button_style='info', layout=widgets.Layout(width='220px'))
out = widgets.Output()

def on_click(b):
    with out:
        clear_output(wait=True)
        row = sample.iloc[idx_slider.value]
        print(f"Selected: {row['productDisplayName']} | "
              f"{row['masterCategory']} > {row.get('articleType','')}")
        visualize_recommendations(idx_slider.value, top_k=k_slider.value)

btn.on_click(on_click)
display(widgets.VBox([widgets.HBox([idx_slider, k_slider]), btn, out]))

---
#  Task 2 — Unique Product Catalog Creation

**Goal:** Automatically detect near-duplicate products and produce a clean catalog with one canonical entry per product.

**Approach:**
1. Compute pairwise cosine distance between all product embeddings
2. Run Agglomerative Clustering with a distance threshold (≈ 0.85 cosine similarity)
3. For each cluster, select the product closest to the centroid as the canonical entry
4. Build and export the deduplicated catalog

In [ ]:
# ── Build pairwise cosine distance matrix ──────────────────────────────────────
# Embeddings are already L2-normalised → dot product = cosine similarity
# Slice to a manageable subset if needed (full 1000x1000 is fine)

cos_sim_matrix  = embeddings @ embeddings.T          # (N, N) cosine similarity
cos_sim_matrix  = np.clip(cos_sim_matrix, -1.0, 1.0) # numerical stability
dist_matrix     = 1.0 - cos_sim_matrix               # cosine distance

print(f'Distance matrix shape: {dist_matrix.shape}')
print(f'Max similarity (self): {cos_sim_matrix.diagonal().max():.4f}')
print(f'Mean pairwise similarity: {cos_sim_matrix[np.triu_indices_from(cos_sim_matrix, k=1)].mean():.4f}')

In [ ]:
# ── Visualize similarity distribution ─────────────────────────────────────────
upper_tri = cos_sim_matrix[np.triu_indices_from(cos_sim_matrix, k=1)]

fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(upper_tri, bins=80, color='#7B1FA2', alpha=0.75, edgecolor='white')
ax.axvline(x=0.85, color='#E53935', linestyle='--', linewidth=2, label='Threshold (0.85)')
ax.set_xlabel('Cosine Similarity', fontsize=12)
ax.set_ylabel('Pair Count', fontsize=12)
ax.set_title('Pairwise Cosine Similarity Distribution', fontsize=13, fontweight='bold', color='#4A148C')
ax.legend(fontsize=11)
ax.set_facecolor('#f8f4ff')
fig.patch.set_facecolor('#f8f4ff')
plt.tight_layout()
plt.savefig('task2_similarity_dist.png', dpi=150, bbox_inches='tight',
            facecolor=fig.get_facecolor())
plt.show()

n_high_sim = (upper_tri > 0.85).sum()
print(f'Pairs with similarity > 0.85 (likely duplicates): {n_high_sim}')

In [ ]:
# ── Agglomerative Clustering ───────────────────────────────────────────────────
DISTANCE_THRESHOLD = 0.15   # corresponds to cosine_similarity > 0.85

clustering = AgglomerativeClustering(
    n_clusters=None,
    distance_threshold=DISTANCE_THRESHOLD,
    metric='precomputed',
    linkage='average'
)
labels = clustering.fit_predict(dist_matrix)

n_clusters    = len(set(labels))
cluster_sizes = pd.Series(labels).value_counts()

print(f'Products in:  {len(sample)}')
print(f'Clusters found: {n_clusters}')
print(f'Catalog reduction: {(1 - n_clusters/len(sample))*100:.1f}%')
print(f'Largest cluster: {cluster_sizes.max()} items')
print(f'Clusters with duplicates (>1 item): {(cluster_sizes > 1).sum()}')
print('\nCluster size distribution:')
print(cluster_sizes.value_counts().sort_index().to_string())

In [ ]:
def get_cluster_representative(cluster_label: int) -> int:
    """Return the index of the product closest to the cluster centroid."""
    member_indices = np.where(labels == cluster_label)[0]
    if len(member_indices) == 1:
        return member_indices[0]
    cluster_embeds = embeddings[member_indices]   # (k, 512)
    centroid       = cluster_embeds.mean(axis=0)
    centroid       = centroid / np.linalg.norm(centroid)
    dists          = np.linalg.norm(cluster_embeds - centroid, axis=1)
    return member_indices[dists.argmin()]


# Build the unique catalog
unique_indices = [get_cluster_representative(c) for c in range(n_clusters)]
unique_catalog = sample.iloc[unique_indices].copy().reset_index(drop=True)
unique_catalog['cluster_id']   = range(n_clusters)
unique_catalog['cluster_size'] = [int((labels == c).sum()) for c in range(n_clusters)]

print(f'\nUnique catalog size: {len(unique_catalog)}')
unique_catalog[['id','productDisplayName','masterCategory','articleType','cluster_size']].head(10)

In [ ]:
def visualize_duplicate_cluster(cluster_label: int, max_show: int = 6):
    """Show all members of a cluster, highlighting the canonical representative."""
    member_indices = np.where(labels == cluster_label)[0]
    rep_idx        = get_cluster_representative(cluster_label)

    n = min(len(member_indices), max_show)
    fig, axes = plt.subplots(1, n, figsize=(3.5 * n, 4))
    fig.patch.set_facecolor('#f8f4ff')
    if n == 1:
        axes = [axes]

    for ax, idx in zip(axes, member_indices[:n]):
        row = sample.iloc[idx]
        try:
            ax.imshow(Image.open(row['img_path']).convert('RGB'))
        except Exception:
            ax.text(0.5, 0.5, 'No image', ha='center', va='center', transform=ax.transAxes)

        is_rep = (idx == rep_idx)
        edge_color = '#4CAF50' if is_rep else '#9E9E9E'
        lw         = 4 if is_rep else 1
        for spine in ax.spines.values():
            spine.set_edgecolor(edge_color)
            spine.set_linewidth(lw)
        ax.set_frame_on(True)

        label = f" CANONICAL\n" if is_rep else ""
        ax.set_title(
            f"{label}{row.get('productDisplayName','')[:22]}\n{row.get('articleType','')}",
            fontsize=7.5, color='#1B5E20' if is_rep else '#424242'
        )
        ax.axis('off')

    plt.suptitle(f'Cluster {cluster_label} — {len(member_indices)} near-duplicate(s)',
                 fontsize=11, fontweight='bold', color='#4A148C')
    plt.tight_layout()
    plt.savefig(f'task2_cluster_{cluster_label}.png', dpi=150, bbox_inches='tight',
                facecolor=fig.get_facecolor())
    plt.show()


# Show the largest duplicate clusters
top_clusters = cluster_sizes[cluster_sizes > 1].head(5).index.tolist()
print(f'Visualising {len(top_clusters)} largest duplicate clusters:\n')
for c in top_clusters:
    visualize_duplicate_cluster(c)

In [ ]:
# ── Stats visualization ────────────────────────────────────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
fig.patch.set_facecolor('#f8f4ff')

# Cluster size distribution
size_counts = cluster_sizes.value_counts().sort_index()
ax1.bar(size_counts.index.astype(str), size_counts.values,
        color='#7B1FA2', edgecolor='white', alpha=0.85)
ax1.set_xlabel('Cluster Size (# duplicates)', fontsize=11)
ax1.set_ylabel('Number of Clusters', fontsize=11)
ax1.set_title('Cluster Size Distribution', fontsize=12, fontweight='bold', color='#4A148C')
ax1.set_facecolor('#f8f4ff')

# Before vs after catalog size by category
before = sample['masterCategory'].value_counts()
after  = unique_catalog['masterCategory'].value_counts()
cats   = before.index
x      = np.arange(len(cats))
w      = 0.35
ax2.bar(x - w/2, before[cats].values, w, label='Before', color='#9E9E9E', alpha=0.8)
ax2.bar(x + w/2, after.reindex(cats, fill_value=0).values, w,
        label='After dedup', color='#4CAF50', alpha=0.85)
ax2.set_xticks(x)
ax2.set_xticklabels(cats, rotation=30, ha='right', fontsize=9)
ax2.set_ylabel('Product Count', fontsize=11)
ax2.set_title('Catalog Size: Before vs After Deduplication', fontsize=12,
               fontweight='bold', color='#4A148C')
ax2.legend(fontsize=10)
ax2.set_facecolor('#f8f4ff')

plt.suptitle(f'Deduplication Results — {len(sample)} → {len(unique_catalog)} products '
             f'({(1 - len(unique_catalog)/len(sample))*100:.1f}% reduction)',
             fontsize=12, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('task2_dedup_stats.png', dpi=150, bbox_inches='tight',
            facecolor=fig.get_facecolor())
plt.show()

# Export catalog to CSV
export_cols = ['id', 'productDisplayName', 'masterCategory', 'subCategory',
               'articleType', 'cluster_id', 'cluster_size']
unique_catalog[export_cols].to_csv('unique_catalog.csv', index=False)
print('\n Unique catalog exported → unique_catalog.csv')

---
#  Task 3 — Reverse Product Search (Text → Image)

**Goal:** Let users search the product catalog using natural language queries instead of images.

**Approach:**  
CLIP maps text and images into the **same** 512-d embedding space. So we encode the query text with CLIP's text encoder and find the nearest image embeddings via cosine similarity — no extra training needed.

In [ ]:
def encode_text_query(text: str) -> np.ndarray:
    """Encode a text string with CLIP. Returns normalised (512,) vector."""
    tokens = clip.tokenize([text]).to(device)
    with torch.no_grad():
        text_embed = model.encode_text(tokens).float()
    text_embed = text_embed / text_embed.norm(dim=-1, keepdim=True)
    return text_embed.cpu().numpy().squeeze()   # (512,)


def text_search(query: str, top_k: int = 5, search_in: pd.DataFrame = None,
                search_embeds: np.ndarray = None):
    """
    Search products by text query.
    Returns list of dicts with product info + similarity score.
    Defaults to searching the full `sample` dataset.
    """
    df   = search_in if search_in is not None else sample
    embs = search_embeds if search_embeds is not None else embeddings

    query_embed = encode_text_query(query)          # (512,)
    scores      = embs @ query_embed                # (N,) cosine similarities
    top_indices = scores.argsort()[-top_k:][::-1]

    results = []
    for rank, idx in enumerate(top_indices):
        r = df.iloc[idx].to_dict()
        r['score'] = float(scores[idx])
        r['rank']  = rank + 1
        r['idx']   = int(idx)
        results.append(r)
    return results


print('Text search engine ready ✓')

In [ ]:
def visualize_search_results(query: str, top_k: int = 5, save: bool = True):
    results = text_search(query, top_k=top_k)

    fig = plt.figure(figsize=(3.5 * top_k, 5.5))
    fig.patch.set_facecolor('#f8f4ff')

    # Header bar
    fig.text(0.5, 0.97,
             f'  "{query}"  —  Top {top_k} Matches',
             ha='center', va='top', fontsize=13, fontweight='bold', color='#4A148C')

    axes = fig.subplots(1, top_k)
    if top_k == 1:
        axes = [axes]

    bar_colors = ['#FFD700', '#C0C0C0', '#CD7F32', '#90CAF9', '#A5D6A7']

    for ax, res in zip(axes, results):
        try:
            ax.imshow(Image.open(res['img_path']).convert('RGB'))
        except Exception:
            ax.text(0.5, 0.5, 'No image', ha='center', va='center', transform=ax.transAxes)

        rank   = res['rank']
        medals = {1: 'I', 2: 'II', 3: 'III'}
        medal  = medals.get(rank, f'#{rank}')

        ax.set_title(
            f"{medal} {res.get('productDisplayName','')[:24]}\n"
            f"{res.get('articleType','')}\n"
            f"Score: {res['score']:.4f}",
            fontsize=7.5, color='#1B5E20'
        )
        ax.axis('off')
        for spine in ax.spines.values():
            spine.set_edgecolor(bar_colors[rank - 1])
            spine.set_linewidth(3)
        ax.set_frame_on(True)

    plt.subplots_adjust(top=0.88)
    if save:
        safe_name = query.replace(' ', '_')[:30]
        plt.savefig(f'task3_search_{safe_name}.png', dpi=150, bbox_inches='tight',
                    facecolor=fig.get_facecolor())
    plt.show()
    return results


# ── Demo queries ───────────────────────────────────────────────────────────────
demo_queries = [
    'blue casual shirt',
    'black running shoes',
    'red floral dress',
    'leather watch men',
    'white sports socks',
]

for q in demo_queries:
    print(f'\n── Query: "{q}" ──')
    results = visualize_search_results(q, top_k=5)
    for r in results:
        print(f"  {r['rank']}. {r.get('productDisplayName','')[:40]:40s} | "
              f"{r.get('articleType',''):20s} | score={r['score']:.4f}")

In [ ]:
# ── Interactive text search widget ─────────────────────────────────────────────
query_box = widgets.Text(
    value='blue casual shirt',
    placeholder='e.g. red floral dress, black leather sneakers …',
    description='Search:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='500px')
)
topk_slider = widgets.IntSlider(
    value=5, min=1, max=10, step=1,
    description='Results:', style={'description_width': 'initial'}
)
search_btn = widgets.Button(
    description=' Search', button_style='success',
    layout=widgets.Layout(width='130px')
)
search_out = widgets.Output()

def on_search(b):
    with search_out:
        clear_output(wait=True)
        q = query_box.value.strip()
        if not q:
            print('Please enter a search query.')
            return
        print(f'Searching for: "{q}" …')
        results = visualize_search_results(q, top_k=topk_slider.value, save=False)
        print('\nTop matches:')
        for r in results:
            print(f"  {r['rank']}. {r.get('productDisplayName','')[:45]:45s} | score={r['score']:.4f}")

search_btn.on_click(on_search)
display(widgets.VBox([
    widgets.HBox([query_box, topk_slider]),
    search_btn,
    search_out
]))

In [ ]:
# ── Cross-modal alignment visualization ───────────────────────────────────────
# Show how text queries align with different product categories
test_queries  = ['shirt', 'shoes', 'watch', 'bag', 'dress', 'jacket']
test_cats     = sample['masterCategory'].unique()[:6]

# Average embedding per category
cat_avg_embeds = {}
for cat in test_cats:
    mask = (sample['masterCategory'] == cat).values
    if mask.sum() > 0:
        avg = embeddings[mask].mean(axis=0)
        cat_avg_embeds[cat] = avg / np.linalg.norm(avg)

# Compute text-to-category similarity matrix
query_embeds_arr = np.array([encode_text_query(q) for q in test_queries])
cat_embeds_arr   = np.array([cat_avg_embeds[c] for c in cat_avg_embeds])
sim_matrix       = query_embeds_arr @ cat_embeds_arr.T

fig, ax = plt.subplots(figsize=(10, 5))
im = ax.imshow(sim_matrix, cmap='Purples', aspect='auto', vmin=0)
ax.set_xticks(range(len(cat_avg_embeds)))
ax.set_xticklabels(list(cat_avg_embeds.keys()), rotation=30, ha='right', fontsize=10)
ax.set_yticks(range(len(test_queries)))
ax.set_yticklabels(test_queries, fontsize=10)
for i in range(len(test_queries)):
    for j in range(len(cat_avg_embeds)):
        ax.text(j, i, f'{sim_matrix[i, j]:.3f}', ha='center', va='center',
                fontsize=8, color='white' if sim_matrix[i, j] > 0.25 else 'black')
plt.colorbar(im, ax=ax, label='Cosine Similarity')
ax.set_title('Text Query ↔ Product Category Alignment (CLIP)',
             fontsize=12, fontweight='bold', color='#4A148C')
plt.tight_layout()
plt.savefig('task3_alignment_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
print('\n Heatmap saved → task3_alignment_heatmap.png')

---
##  Final Summary

| Task | Method | Key Result |
|------|--------|------------|
| 1 — Recommendations | CLIP image embeddings + complementary category map + cosine similarity | Contextually relevant cross-category suggestions |
| 2 — Deduplication | Agglomerative clustering on CLIP cosine distances (threshold 0.85) | ~10–30% catalog reduction, centroid-based representative |
| 3 — Text Search | CLIP text encoder → cosine search over image embedding index | Natural language → matching products, zero training |

In [ ]:
# ── Save all outputs ──────────────────────────────────────────────────────────
import zipfile, glob

output_files = glob.glob('*.png') + glob.glob('*.csv')
with zipfile.ZipFile('all_outputs.zip', 'w') as zf:
    for f in output_files:
        zf.write(f)

print('All outputs packaged → all_outputs.zip')
print('Files included:')
for f in sorted(output_files):
    print(f'  {f}')

# Download zip in Colab
from google.colab import files
files.download('all_outputs.zip')